In [3]:
from langchain_groq import ChatGroq
from typing import List, TypedDict
import time
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
import os
load_dotenv(override=True)
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
print(f"API Key Loaded: {bool(os.getenv('GROQ_API_KEY'))}")



API Key Loaded: True


In [4]:
docs = (
    PyPDFLoader("./documents/book1.pdf").load() +
    PyPDFLoader("./documents/book2.pdf").load() +
    PyPDFLoader("./documents/book3.pdf").load()
)

In [5]:
len(docs)

2123

In [6]:
chunks = RecursiveCharacterTextSplitter(chunk_size=900,chunk_overlap=200).split_documents(docs)

for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [7]:
len(chunks)

6637

In [ ]:
# 3) Index (fresh collection each run)
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_Store.as_retriever(search_type='similarity',search_kwargs={'k':4})


In [ ]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
class State(TypedDict):
    question:str
    docs:List[Document]
    ans:str

In [ ]:
def retrieve(state):
    q=state[question]
    return {"docs": retriever.invoke(q)}
    

In [ ]:
prompt=ChatPromptTemplat.from_message(
  ("system", "Answer only from the context. If not in context, say you don't know."),
        ("human", "Question: {question}\n\nContext:\n{context}"),
        )

def generate(state):
    context = "\n\n".join(d.page_content for d in state["docs"])
    out = (prompt | llm).invoke({"question":state["question","context":context]})
    return {"answer" : out.content}

In [ ]:
g = StateGraph(State)
g.add_node("retrieve", retrieve)
g.add_node("generate", generate)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", END)
app = g.compile()

app

In [ ]:
res = app.invoke({"question": "WHat is a transformer in deep learning.", "docs": [], "answer": ""})
print(res["answer"])

In [ ]:
rint(res['docs'][0].page_content)
print('*'*100)
print(res['docs'][1].page_content)
print('*'*100)
print(res['docs'][2].page_content)
print('*'*100)
print(res['docs'][3].page_content)